# 03 NLP: plot text, similarity and validation (Person B)

## 1. Cleaning the plots and masking character names

Character names ("Cobb", "Sarah") would make two plots look similar when the stories are not, so every
NLP step starts from a version of the plots with the names removed. `nlp/scripts/clean_plots.py` does
this once and writes `nlp/data/plots_clean.tsv`:

| column | used for |
|---|---|
| `text_masked` | sentence encoders: the original text, every character name replaced by `PERSON` |
| `tokens` | TF-IDF and LDA: lowercased lemmas, no stopwords / punctuation / numbers / names |
| `tokens_names_kept` | robustness check: the same tokens with the names left in |

**Decisions.** Only PERSON entities are masked; places and organisations ("FBI", "Vietnam", "Mars") are
story content and stay. The NER model is spaCy `en_core_web_sm` (as in week 7, exercise 7.8). Titles and
roles ("Agent", "Captain", "Mom") are kept.

**Three rules**, because `en_core_web_sm` misses many names on its own:
1. tokens inside a PERSON entity;
2. *name spreading* within one plot: once "Dom Cobb" is found, "Dom" and "Cobb" are masked everywhere in that plot;
3. a corpus-wide *name lexicon*: the small model often labels a character ORG for a whole plot
   ("Sophie": ORG 171 times, PERSON once), so a proper noun is also masked when the corpus as a whole votes it a name.

Run `python nlp/scripts/clean_plots.py` from the repository root first (about 6 minutes the first time, seconds afterwards: the spaCy parse is cached in `nlp/cache/`).

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

# The notebook runs in nlp/notebooks/; the project code is in nlp/scripts/
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import paths               # every file location (nlp/scripts/paths.py)
import clean_plots as cp   # step 1 code: the name-masking rules

# ids:   list of 1744 node_ids, in films_nodes.tsv order (the row order of every file)
# plots: dict node_id -> raw plot text
ids, plots = cp.load_plots()

# docs: list of 1744 parsed plots (spaCy Doc objects), same order as ids. Loaded from nlp/cache/.
docs, nlp = cp.parse(ids, plots)
doc_of = dict(zip(ids, docs))                  # node_id -> parsed plot

stopwords = nlp.Defaults.stop_words            # set of ~330 lowercase English stopwords
lexicon = cp.name_lexicon(docs, stopwords)     # rule 3: set of ~7,700 words that are usually names

# clean: the output of clean_plots.py. A table with 1744 rows (one per film) and the columns
#        node_id, text_masked, tokens, tokens_names_kept, n_names_masked
clean = pd.read_csv(paths.PLOTS_CLEAN, sep="\t", quoting=3, keep_default_na=False)
assert list(clean.node_id) == ids, "plots_clean.tsv is not in films_nodes.tsv order"

print(len(clean), "films |", len(lexicon), "words in the name lexicon")
clean[["node_id", "n_names_masked"]].describe()

cache hit: docs-en_core_web_sm-3.8.0-spacy3.8.16-c8386c35dfa7.spacy


1744 films | 7557 words in the name lexicon


,n_names_masked
count,1744.000000
mean,62.463303
std,31.611850
min,0.000000
25%,45.000000
50%,61.000000
75%,78.000000
max,559.000000


### 1.1 Before and after

In [2]:
clean_by_film = clean.set_index("node_id")     # look up a film's row by its node_id

for film in ["Inception", "La_La_Land", "Gravity_(2013_film)"]:
    print(f"=== {film}")
    print(f"--- original\n{plots[film][:500]}")                       # first 500 characters
    print(f"--- text_masked\n{clean_by_film.text_masked[film][:500]}")
    print(f"--- tokens\n{clean_by_film.tokens[film][:300]}\n")

=== Inception
--- original
Dom Cobb and Arthur are professional thieves who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets' subconscious and extract information. Their latest target Saito is impressed with Cobb's ability to layer multiple dreams within each other. He offers to hire Cobb for the ostensibly impossible job of performing "inception": implanting Robert Fischer, the son of Saito's competitor Maurice Fischer, with the idea to dissolve his father's conglomerate
--- text_masked
PERSON and PERSON are professional thieves who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets' subconscious and extract information. Their latest target PERSON is impressed with PERSON's ability to layer multiple dreams within each other. He offers to hire PERSON for the ostensibly impossible job of performing "inception": implanting PERSON, the son of PERSON's competitor PERSON, with the idea to dis

### 1.2 What each rule adds

Number of masked tokens with rule 1 only, rules 1-2, and all three rules.

In [3]:
# Count the masked tokens over all 1744 plots, adding one rule at a time.
# cp.name_mask returns (mask, persons). mask is a bool array with one value per token, so mask.sum()
# is the number of masked tokens in that plot.
masked_tokens = {"rule 1 (spaCy PERSON)": 0, "+ rule 2 (spreading)": 0, "+ rule 3 (lexicon)": 0}

for doc in docs:
    mask_rule1, _ = cp.name_mask(doc, stopwords, spread_names=False)
    mask_rules12, _ = cp.name_mask(doc, stopwords)
    mask_rules123, _ = cp.name_mask(doc, stopwords, lexicon)
    masked_tokens["rule 1 (spaCy PERSON)"] += mask_rule1.sum()
    masked_tokens["+ rule 2 (spreading)"] += mask_rules12.sum()
    masked_tokens["+ rule 3 (lexicon)"] += mask_rules123.sum()

pd.Series(masked_tokens, name="masked tokens")

rule 1 (spaCy PERSON)     78457
+ rule 2 (spreading)     104071
+ rule 3 (lexicon)       108936
Name: masked tokens, dtype: int32

### 1.3 What is left: capitalised words not at the start of a sentence

Read this list for missed names. Expected leftovers are places ("New York"), nationalities, titles and
holidays; names in this list are misses.

In [4]:
# A capitalised word that does not start a sentence is probably a proper noun.
#   (?<![.!?]\s)  not right after the end of a sentence
#   (?<!^)        not at the very start of the text
#   [A-Z][a-z]+   a capitalised word
CAPITALISED_MID_SENTENCE = re.compile(r"(?<![.!?]\s)(?<!^)\b([A-Z][a-z]+)\b")

occurrences = Counter()    # word -> number of occurrences in all masked plots
films_with = {}            # word -> set of films it occurs in

for film, text in zip(clean.node_id, clean.text_masked):
    for match in CAPITALISED_MID_SENTENCE.finditer(text):
        word = match.group(1)
        occurrences[word] += 1
        films_with.setdefault(word, set()).add(film)

# The 60 most frequent leftover capitalised words
leftovers = [(word, count, len(films_with[word])) for word, count in occurrences.most_common(60)]
pd.DataFrame(leftovers, columns=["word", "count", "films"])

,word,count,films
0,The,583,278
1,New,560,362
2,York,392,258
3,American,286,212
4,Dr,270,145
5,City,231,186
6,Mr,220,75
7,Earth,216,80
8,Los,205,149
9,Angeles,204,145


### 1.4 Validation against hand labels (as in exercise 7.8)

`nlp/validation/ner_gold_sample.tsv` holds 40 random plot sentences (seed 42). **Fill in the `persons` column by hand
before looking at the masks**: every character name as written in the sentence, separated by ` | `
(e.g. `Hal | Oliver | Georgia`). Leave it empty if the sentence has no names. Real people (e.g. "Richard Nixon")
count as names too; titles ("Captain") do not.

Scoring is per token: a gold name token is found if it is masked. **Recall matters most**: a name left in the
text creates fake similarity, while an extra masked word costs little.

In [5]:
# gold: one row per hand-labelled sentence (40 in the file). Columns: node_id, start, end (character offsets
#       of the sentence in the plot), sentence, persons (the hand label, e.g. "Hal | Oliver")
gold = pd.read_csv(paths.NER_GOLD, sep="\t", quoting=3, keep_default_na=False)
# Sentences from films that are no longer in the network (removed as sequels) cannot be scored
in_network = gold.node_id.isin(doc_of)
if not in_network.all():
    print(f"skipping {(~in_network).sum()} hand-labelled sentences from removed films:",
          ", ".join(gold.node_id[~in_network]))
gold = gold[in_network]


def gold_name_tokens(doc, start, end, persons):
    """The token positions of the hand-labelled names in one sentence.

    Every occurrence of every labelled name counts. Titles ("Captain") are not counted as name tokens,
    the same as in the masking rules. Returns a set of token indices into doc.
    """
    sentence = doc.text[start:end]
    positions = set()
    names = [p.strip() for p in persons.split("|") if p.strip()]
    for name in names:
        # whole-word matches of the name inside the sentence
        for match in re.finditer(r"(?<!\w)" + re.escape(name) + r"(?!\w)", sentence):
            span = doc.char_span(start + match.start(), start + match.end(), alignment_mode="expand")
            positions |= {t.i for t in span if t.is_alpha and t.text not in cp.TITLES}
    return positions


def score(**mask_options):
    """Token-level precision and recall of the masking on the 40 hand-labelled sentences.

    mask_options go to cp.name_mask: spread_names=False (rule 1 only), nothing (rules 1-2),
    or lexicon=lexicon (rules 1-3).
        precision = correct / masked   how many masked tokens really are names
        recall    = correct / gold     how many real name tokens were masked
    """
    n_correct = n_masked = n_gold = 0
    for row in gold.itertuples():
        doc = doc_of[row.node_id]
        mask, _ = cp.name_mask(doc, stopwords, **mask_options)
        sentence = doc.char_span(row.start, row.end, alignment_mode="expand")
        masked = {t.i for t in sentence if mask[t.i]}                      # what the code masked
        true_names = gold_name_tokens(doc, row.start, row.end, row.persons)  # what the hand label says
        n_correct += len(masked & true_names)
        n_masked += len(masked)
        n_gold += len(true_names)
    return {"masked": n_masked, "gold": n_gold, "correct": n_correct,
            "precision": round(n_correct / n_masked, 3) if n_masked else None,
            "recall": round(n_correct / n_gold, 3) if n_gold else None}


if (gold.persons == "").all():
    print("ner_gold_sample.tsv has no labels yet: fill in the persons column, then rerun this cell.")
else:
    display(pd.DataFrame({
        "rule 1 only": score(spread_names=False),
        "rules 1-2": score(),
        "rules 1-3": score(lexicon=lexicon),
    }).T)

skipping 2 hand-labelled sentences from removed films: The_Purge:_Anarchy, The_Purge:_Election_Year


,masked,gold,correct,precision,recall
rule 1 only,64.0,86.0,61.0,0.953,0.709
rules 1-2,78.0,86.0,75.0,0.962,0.872
rules 1-3,82.0,86.0,79.0,0.963,0.919


**Reading the result.** If recall with all three rules is clearly below ~0.9, look at the missed names
(print `true_names - masked` per sentence) before moving on to TF-IDF: either extend the rules or switch to a larger
spaCy model. Do not tune the rules on these 40 sentences and then report the same 40 as the validation.

## 2. Plot similarity: four methods

`nlp/scripts/build_similarity.py` turns every cleaned plot into a vector, once per method. Similarity between two films is the
cosine of their vectors, so every method gives a 1744 x 1744 matrix in the order of `films_nodes.tsv`.

| method | input | what it captures | course |
|---|---|---|---|
| `tfidf` | `tokens` | shared distinctive words | 6.2 |
| `lda` | `tokens` | shared topic mixture (K = 20) | 6.5 |
| `minilm` | `text_masked` | meaning of the text, small encoder (all-MiniLM-L6-v2) | 7.2 |
| `mpnet` | `text_masked` | the same, larger encoder (all-mpnet-base-v2) | 7.2 |
| `tfidf_names_kept` | `tokens_names_kept` | TF-IDF without name masking (robustness only) | |

Decisions:
- **TF-IDF.** Words must appear in at least 2 films and at most half of them.
- **LDA.** Same vocabulary filter as 6.5 (min 5 films, max half). K = 20 topics, seed 0, 50 iterations.
- **Encoders.**
  - Plots are split into passages of whole sentences, at most 180 words each, as in 7.2.
  - Both encoders read the same passages, so they differ only in the model.
  - A film vector is the mean of its passage vectors.

Run `python nlp/scripts/build_similarity.py` first, with the Python that has `sentence-transformers` (3.13 here). This notebook only reads the saved vectors, so any kernel with numpy, scipy and pandas works.

For the network comparison, any other notebook can load a matrix in one line:
```python
S = np.load("../nlp/similarity/sim_embed.npy")   # 1744 x 1744, rows/columns in films_nodes.tsv order
```

In [6]:
import numpy as np
from scipy.stats import spearmanr
import build_similarity as bs     # step 2 code (nlp/scripts/build_similarity.py)

METHODS = ["tfidf", "lda", "minilm", "mpnet"]
assert bs.node_ids() == list(clean.node_id) == ids

# S[m]: the film-film cosine similarity of method m, a numpy array 1744 x 1744.
#       S[m][i, j] = similarity of the films in rows i and j. The diagonal is 1.
S = {m: bs.similarity(m) for m in METHODS + ["tfidf_names_kept"]}

n = len(ids)                                    # 1744 films
# Every film pair once (i < j): two int arrays of length 1,519,896 (the row and the column of each pair)
iu = np.triu_indices(n, 1)
# pairs[m]: the similarity of every film pair under method m, 1,519,896 values in the order of iu
pairs = {m: S[m][iu] for m in S}

# nodes: films_nodes.tsv, 1744 rows (same order as ids)
nodes = pd.read_csv(paths.FILMS_NODES, sep="\t", comment="#", quoting=3, keep_default_na=False)
assert list(nodes.node_id) == ids
title = dict(zip(nodes.node_id, nodes.name))           # node_id -> film title
genres = [set(g.split(",")) for g in nodes.genres]     # list of 1744 sets, e.g. {"Drama", "Sci-Fi"}

print(f"{n} films, {len(iu[0]):,} pairs")

1744 films, 1,519,896 pairs


### 2.1 How the similarities are distributed

The methods use very different scales. Encoders put almost every pair between 0.2 and 0.6, while TF-IDF puts most
pairs near 0. So the raw values cannot be compared across methods. **All comparisons below use ranks.**

In [7]:
# Summary statistics of the 1.6M pair similarities of each method (one row per method)
distribution = {m: pd.Series(pairs[m]).describe(percentiles=[0.5, 0.9, 0.99]) for m in METHODS}
pd.DataFrame(distribution).T.round(3)

,count,mean,std,min,50%,90%,99%,max
tfidf,1519896.0,0.029,0.020,0.000,0.024,0.052,0.097,0.847
lda,1519896.0,0.346,0.298,0.000,0.286,0.807,0.974,1.000
minilm,1519896.0,0.517,0.128,-0.082,0.525,0.678,0.774,0.898
mpnet,1519896.0,0.611,0.117,0.021,0.622,0.754,0.835,0.931


### 2.2 Do the methods agree?

Two views:
- **Spearman correlation** over all 1.6M film pairs: do two methods rank pairs the same way overall?
- **Top-10 neighbour overlap**: of each film's 10 most similar films under method A, how many are also in its top 10
  under method B? This view matters most for the project, because closest neighbours drive any comparison with the
  network. By chance the overlap would be about 10/1743 = 0.6%.

In [8]:
def top_k_neighbours(S_m, k=10):
    """The k most similar films of every film under one similarity matrix.

    S_m: 1744 x 1744 similarity matrix.
    Returns an int array 1744 x k: row i holds the row numbers of film i's k nearest films, most similar
    first. A film is never its own neighbour (its diagonal value is set to minus infinity first).
    """
    S_without_self = S_m.copy()
    np.fill_diagonal(S_without_self, -np.inf)
    return np.argsort(-S_without_self, axis=1)[:, :k]


def neighbour_overlap(top_a, top_b):
    """Average share of neighbours two lists have in common: 1 = identical, 0 = nothing shared.

    top_a, top_b: 1744 x k arrays from top_k_neighbours.
    """
    k = top_a.shape[1]
    return np.mean([len(set(row_a) & set(row_b)) / k for row_a, row_b in zip(top_a, top_b)])


top = {m: top_k_neighbours(S[m]) for m in S}      # top[m]: 1744 x 10, each film's 10 nearest films

spearman = pd.DataFrame(index=METHODS, columns=METHODS, dtype=float)   # 4 x 4
overlap = pd.DataFrame(index=METHODS, columns=METHODS, dtype=float)    # 4 x 4
for a in METHODS:
    for b in METHODS:
        spearman.loc[a, b] = 1.0 if a == b else spearmanr(pairs[a], pairs[b])[0]
        overlap.loc[a, b] = neighbour_overlap(top[a], top[b])

print("Spearman correlation over all film pairs")
display(spearman.round(2))
print("Share of top-10 neighbours in common")
display(overlap.round(2))

Spearman correlation over all film pairs


,tfidf,lda,minilm,mpnet
tfidf,1.00,0.45,0.60,0.57
lda,0.45,1.00,0.51,0.48
minilm,0.60,0.51,1.00,0.90
mpnet,0.57,0.48,0.90,1.00


Share of top-10 neighbours in common


,tfidf,lda,minilm,mpnet
tfidf,1.00,0.06,0.18,0.18
lda,0.06,1.00,0.09,0.08
minilm,0.18,0.09,1.00,0.44
mpnet,0.18,0.08,0.44,1.00


### 2.3 A sanity check: do neighbours share a genre?

Genre is not the target. The project asks whether plots reveal more than genre. But a working method should still
put films of the same genre close together more often than chance. Compare each method with random pairs.

In [9]:
def shares_genre(i, j):
    """True if the films in rows i and j have at least one IMDb genre in common."""
    return bool(genres[i] & genres[j])


# Baseline: 20,000 random film pairs (pairs of a film with itself are skipped)
rng = np.random.default_rng(0)
random_pairs = rng.integers(0, n, size=(20000, 2))      # 20000 x 2 row numbers
baseline = np.mean([shares_genre(i, j) for i, j in random_pairs if i != j])

# Each method: all 1744 x 10 (film, neighbour) pairs
genre_share = {m: np.mean([shares_genre(i, j) for i in range(n) for j in top[m][i]]) for m in S}

table = pd.Series({"random pairs": baseline, **genre_share}, name="top-10 neighbours sharing a genre")
table.round(3).to_frame()

,top-10 neighbours sharing a genre
random pairs,0.615
tfidf,0.803
lda,0.846
minilm,0.871
mpnet,0.878
tfidf_names_kept,0.670


### 2.4 Nearest neighbours of a few films

Read these lists. They show what each method treats as "similar".

In [10]:
def neighbours_table(film, k=6):
    """The k nearest films of one film under every method, with their cosine. One column per method."""
    i = ids.index(film)
    columns = {}
    for m in METHODS:
        columns[m] = [f"{title[ids[j]]} ({S[m][i, j]:.2f})" for j in top[m][i][:k]]
    return pd.DataFrame(columns)


for film in ["Inception", "La_La_Land", "Gravity_(2013_film)"]:
    print(title[film])
    display(neighbours_table(film))

Inception


,tfidf,lda,minilm,mpnet
0,Dream House (0.13),Wall Street: Money Never Sleeps (0.81),The Dark Tower (0.75),Flatliners (0.75)
1,Columbus (0.11),The Founder (0.81),Insidious (0.71),2:22 (0.75)
2,Total Recall (0.11),On the Basis of Sex (0.81),Get Out (0.70),A Nightmare on Elm Street (0.73)
3,A Nightmare on Elm Street (0.09),Tomorrowland (0.81),A Nightmare on Elm Street (0.69),Chronicle (0.73)
4,Area 51 (0.09),Mississippi Grind (0.78),2:22 (0.69),The 9th Life of Louis Drax (0.73)
5,Think like a Man (0.08),A Most Violent Year (0.78),Swiss Army Man (0.69),Lights Out (0.73)


La La Land


,tfidf,lda,minilm,mpnet
0,Whiplash (0.28),Top Five (0.99),I Can Only Imagine (0.78),Begin Again (0.82)
1,The Runaways (0.20),We Are Your Friends (0.97),Get Him to the Greek (0.78),Breathe In (0.81)
2,Don't Think Twice (0.19),Don't Think Twice (0.97),We Are Your Friends (0.77),Beyond the Lights (0.81)
3,I Can Only Imagine (0.17),"Juliet, Naked (0.96)",Country Strong (0.77),We Are Your Friends (0.81)
4,The Disaster Artist (0.16),Get Him to the Greek (0.96),A Rainy Day in New York (0.77),A Rainy Day in New York (0.81)
5,Sing Street (0.16),The Big Sick (0.96),Rudderless (0.75),Rocketman (0.80)


Gravity


,tfidf,lda,minilm,mpnet
0,Life (0.47),Life (0.99),Life (0.79),Life (0.84)
1,The Martian (0.19),Ad Astra (0.94),First Man (0.64),Passengers (0.70)
2,Ad Astra (0.18),Prometheus (0.94),Geostorm (0.62),Prometheus (0.67)
3,Io (0.18),The Martian (0.94),After Earth (0.61),Europa Report (0.67)
4,First Man (0.16),Pacific Rim (0.93),The Martian (0.61),After Earth (0.66)
5,Geostorm (0.16),The 33 (0.92),Interstellar (0.60),First Man (0.66)


### 2.5 What masking the names changes (TF-IDF)

Compare TF-IDF with and without name masking. A pair that becomes much more similar when names are kept, without
being similar in plot, is exactly the fake similarity step 1 removes. Sequels and remakes that share character
names are the obvious case.

In [11]:
unchanged = neighbour_overlap(top["tfidf"], top["tfidf_names_kept"])
print(f"top-10 neighbours unchanged by masking: {unchanged:.0%}")

# The set of words of every film (lists of 1744 sets), with and without the names
words_names_kept = [set(tokens.split()) for tokens in clean.tokens_names_kept]
words_masked = [set(tokens.split()) for tokens in clean.tokens]

# How much each pair's similarity rises when the names are kept: 1,519,896 values, in the order of iu
gain = pairs["tfidf_names_kept"] - pairs["tfidf"]

rows = []
for p in np.argsort(-gain)[:12]:                  # the 12 pairs that gain the most
    i, j = iu[0][p], iu[1][p]                     # the row numbers of the two films
    # words both films share only in the unmasked version = the names they share
    shared_names = (words_names_kept[i] & words_names_kept[j]) - words_masked[i] - words_masked[j]
    rows.append({"film A": title[ids[i]], "film B": title[ids[j]],
                 "masked": round(S["tfidf"][i, j], 2),
                 "names kept": round(S["tfidf_names_kept"][i, j], 2),
                 "shared names": " ".join(sorted(shared_names)[:8])})
pd.DataFrame(rows)

top-10 neighbours unchanged by masking: 12%


,film A,film B,masked,names kept,shared names
0,A Monster Calls,Sing Street,0.02,0.77,conor
1,A Haunted House,Dope,0.04,0.79,malcolm
2,Beautiful Boy,The Signal,0.04,0.77,nic
3,Celeste and Jesse Forever,Vox Lux,0.03,0.76,celeste
4,Eli,The Book of Eli,0.02,0.72,eli
5,Fallen,Luce,0.04,0.74,luce
6,Holmes & Watson,Sherlock Holmes: A Game of Shadows,0.10,0.80,holmes james john lestrade moriarty mycroft wa...
7,Afflicted,Zoolander 2,0.04,0.73,derek
8,Me Before You,Nightcrawler,0.02,0.70,lou
9,Clown,The Numbers Station,0.02,0.71,kent


### 2.6 LDA topics

The top words of each topic, and its average weight over all films. Topics with leftover names (e.g. "amanda",
"freddie") show the names step 1 missed. They also show that a few films with many uses of one name can form a
topic of their own.

In [12]:
# One row per topic: its number, its average weight over all films, and its 15 most probable words
pd.read_csv(paths.LDA_TOPICS, sep="\t")

,topic,share,top_words
0,0,0.0413,film new work york scene story movie play city...
1,1,0.0455,child family camp town war baby farm save daug...
2,2,0.0379,alien earth destroy escape human attack soldie...
3,3,0.0186,hotel book write man paris novel room woman re...
4,4,0.0155,school student game class player high teacher ...
5,5,0.0683,agent team cia attack president force officer ...
6,6,0.0175,crew boat ship water captain island rescue rob...
7,7,0.0335,body father memory ghost spirit possess child ...
8,8,0.1027,house car escape shoot room attack man see att...
9,9,0.0147,pope shoot zoo save cabin daughter police atte...
